# Correlation Analysis: News Sentiment and Stock Movement
## Task 3: Analyzing the Relationship Between News Sentiment and Stock Returns

This notebook performs comprehensive correlation analysis between news sentiment and stock price movements:

### Key Tasks:
1. **Date Alignment**: Normalize timestamps between news and stock datasets
2. **Sentiment Analysis**: Quantify sentiment of news headlines using VADER
3. **Stock Returns Calculation**: Compute daily percentage changes in stock prices
4. **Correlation Analysis**: Calculate Pearson correlation between sentiment scores and stock returns

### Analysis Approach:
- Aggregate daily sentiment scores when multiple articles appear on the same day
- Align news dates with corresponding stock trading days
- Calculate correlation coefficients and statistical significance
- Visualize relationships between sentiment and returns

### Stocks Analyzed:
- AAPL (Apple Inc.)
- AMZN (Amazon.com Inc.)
- GOOG (Alphabet Inc.)
- META (Meta Platforms Inc.)
- MSFT (Microsoft Corporation)
- NVDA (NVIDIA Corporation)


## 1. Setup and Imports


In [ ]:
# Standard library imports
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import numpy as np

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Statistical analysis
from scipy.stats import pearsonr, spearmanr
from scipy import stats

# Set project root and add to path
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Project modules
from src.data_loader import DataLoader, DataPreprocessor
from src.sentiment_analyzer import SentimentAnalyzer, SentimentReturnLinker
from src.stock_data_loader import StockDataLoader

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("="*60)
print("Correlation Analysis Environment Setup Complete")
print("="*60)
print(f"Working directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Data directory: {project_root / 'data'}")


## 2. Load News Headlines Data


In [ ]:
# Load news headlines data
data_path = project_root / 'data' / 'raw_analyst_ratings.csv'

print("Loading news headlines data...")
loader = DataLoader(data_path, chunk_size=100000)
news_df = loader.load_data(show_progress=True)

print(f"\n✓ Loaded {len(news_df):,} news headlines")
print(f"Columns: {list(news_df.columns)}")
print(f"\nFirst few rows:")
print(news_df.head())


## 3. Date Normalization and Alignment


In [ ]:
# Preprocess news data: convert dates and extract temporal features
print("Preprocessing news data...")
preprocessor = DataPreprocessor(news_df)
news_df = (preprocessor
           .convert_dates()
           .extract_temporal_features()
           .get_dataframe())

# Normalize dates: extract date only (remove time component)
# This ensures we can match news dates with stock trading days
news_df['date_only'] = pd.to_datetime(news_df['date']).dt.date
news_df['date_only'] = pd.to_datetime(news_df['date_only'])

# Filter to only stocks we have price data for
available_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'MSFT', 'NVDA']
news_df = news_df[news_df['stock'].isin(available_stocks)].copy()

print(f"\n✓ Processed {len(news_df):,} news headlines")
print(f"Date range: {news_df['date_only'].min()} to {news_df['date_only'].max()}")
print(f"\nStocks in dataset:")
print(news_df['stock'].value_counts().sort_index())
print(f"\nSample of processed data:")
print(news_df[['headline', 'date_only', 'stock']].head(10))


## 4. Sentiment Analysis on News Headlines


In [ ]:
# Initialize sentiment analyzer
print("Initializing sentiment analyzer...")
sentiment_analyzer = SentimentAnalyzer(compound_threshold=0.05)

# Perform sentiment analysis on headlines
print("\nAnalyzing sentiment for all headlines...")
print("This may take a few minutes for large datasets...")

sentiment_scores = sentiment_analyzer.analyze_series(
    news_df['headline'], 
    show_progress=True
)

# Add sentiment scores to news dataframe
news_df = pd.concat([news_df, sentiment_scores], axis=1)

# Get sentiment summary
sentiment_summary = sentiment_analyzer.get_sentiment_summary(sentiment_scores)
print("\n" + "="*60)
print("Sentiment Analysis Summary")
print("="*60)
for key, value in sentiment_summary.items():
    if 'pct' in key:
        print(f"{key:25s}: {value:6.2f}%")
    elif 'count' in key:
        print(f"{key:25s}: {value:10,}")
    else:
        print(f"{key:25s}: {value:10.4f}")

print(f"\n✓ Sentiment analysis complete")
print(f"\nSample headlines with sentiment:")
sample_df = news_df[['headline', 'compound', 'sentiment_label', 'stock']].head(10)
print(sample_df.to_string(index=False))


## 5. Load Stock Price Data


In [ ]:
# Load stock price data for all available stocks
stock_loader = StockDataLoader()
stock_data = {}

print("Loading stock price data...")
for ticker in available_stocks:
    stock_file = project_root / 'data' / f'{ticker}.csv'
    if stock_file.exists():
        print(f"  Loading {ticker}...")
        df = stock_loader.load_from_csv(stock_file, date_col='Date')
        df = stock_loader.prepare_ohlcv_data(df)
        
        # Reset index to have Date as a column for merging
        df = df.reset_index()
        df['Date'] = pd.to_datetime(df['Date'])
        df['date_only'] = pd.to_datetime(df['Date'].dt.date)
        df['stock'] = ticker
        
        stock_data[ticker] = df
        print(f"    ✓ Loaded {len(df):,} trading days")
        print(f"    Date range: {df['date_only'].min()} to {df['date_only'].max()}")

# Combine all stock data
all_stocks_df = pd.concat(stock_data.values(), ignore_index=True)
print(f"\n✓ Total stock trading days: {len(all_stocks_df):,}")
print(f"\nSample stock data:")
print(all_stocks_df[['Date', 'date_only', 'stock', 'Close']].head(10))


## 6. Calculate Daily Stock Returns


In [ ]:
# Initialize sentiment return linker for calculating returns
linker = SentimentReturnLinker()

# Calculate daily returns for each stock
print("Calculating daily stock returns...")
returns_data = []

for ticker in available_stocks:
    if ticker in stock_data:
        stock_df = stock_data[ticker].copy()
        stock_df = stock_df.sort_values('date_only')
        
        # Calculate simple returns (percentage change)
        stock_df['daily_return'] = linker.calculate_returns(
            stock_df['Close'], 
            method='simple'
        )
        
        # Calculate log returns (for statistical analysis)
        stock_df['log_return'] = linker.calculate_returns(
            stock_df['Close'], 
            method='log'
        )
        
        returns_data.append(stock_df[['date_only', 'stock', 'Close', 'daily_return', 'log_return']])
        
        print(f"  {ticker}: {stock_df['daily_return'].notna().sum():,} valid returns")
        print(f"    Mean return: {stock_df['daily_return'].mean():.4f} ({stock_df['daily_return'].mean()*100:.2f}%)")
        print(f"    Std return:  {stock_df['daily_return'].std():.4f} ({stock_df['daily_return'].std()*100:.2f}%)")

# Combine all returns
returns_df = pd.concat(returns_data, ignore_index=True)
print(f"\n✓ Calculated returns for {len(returns_df):,} trading days")
print(f"\nSample returns data:")
print(returns_df.head(10))


## 7. Aggregate Daily Sentiment Scores


In [ ]:
# Aggregate sentiment scores by date and stock
# When multiple articles appear on the same day, calculate average sentiment
print("Aggregating daily sentiment scores...")

daily_sentiment = news_df.groupby(['date_only', 'stock']).agg({
    'compound': ['mean', 'std', 'count'],
    'pos': 'mean',
    'neu': 'mean',
    'neg': 'mean',
    'sentiment_label': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'neutral'
}).reset_index()

# Flatten column names
daily_sentiment.columns = [
    'date_only', 'stock', 
    'avg_sentiment', 'sentiment_std', 'article_count',
    'avg_pos', 'avg_neu', 'avg_neg', 'dominant_sentiment'
]

# Fill NaN std values with 0 (when only one article per day)
daily_sentiment['sentiment_std'] = daily_sentiment['sentiment_std'].fillna(0)

print(f"\n✓ Aggregated sentiment for {len(daily_sentiment):,} date-stock combinations")
print(f"Total articles: {daily_sentiment['article_count'].sum():,}")
print(f"Average articles per day-stock: {daily_sentiment['article_count'].mean():.2f}")
print(f"\nDistribution of articles per day:")
print(daily_sentiment['article_count'].describe())
print(f"\nSample aggregated sentiment:")
print(daily_sentiment.head(10))


## 8. Align Sentiment with Stock Returns


In [ ]:
# Merge sentiment data with returns data
# We'll align sentiment on day T with returns on day T (same-day correlation)
# and also test lagged correlations (sentiment on day T with returns on day T+1)

print("Aligning sentiment with stock returns...")

# Same-day alignment (sentiment and returns on the same day)
aligned_same_day = pd.merge(
    daily_sentiment,
    returns_df[['date_only', 'stock', 'daily_return', 'log_return']],
    on=['date_only', 'stock'],
    how='inner'
)

# Remove rows with missing returns
aligned_same_day = aligned_same_day.dropna(subset=['daily_return', 'avg_sentiment'])

print(f"\n✓ Same-day alignment: {len(aligned_same_day):,} matched observations")
print(f"Date range: {aligned_same_day['date_only'].min()} to {aligned_same_day['date_only'].max()}")

# Lagged alignment (sentiment on day T with returns on day T+1)
# Shift returns forward by 1 day
returns_lagged = returns_df.copy()
returns_lagged['sentiment_date'] = returns_lagged['date_only'] - pd.Timedelta(days=1)

aligned_lagged = pd.merge(
    daily_sentiment,
    returns_lagged[['sentiment_date', 'stock', 'daily_return', 'log_return']],
    left_on=['date_only', 'stock'],
    right_on=['sentiment_date', 'stock'],
    how='inner'
)

aligned_lagged = aligned_lagged.dropna(subset=['daily_return', 'avg_sentiment'])
aligned_lagged = aligned_lagged.drop(columns=['sentiment_date'])

print(f"✓ Lagged alignment (1-day): {len(aligned_lagged):,} matched observations")

print(f"\nSample aligned data (same-day):")
print(aligned_same_day[['date_only', 'stock', 'avg_sentiment', 'daily_return', 'article_count']].head(10))


## 9. Correlation Analysis


In [ ]:
# Calculate correlation between sentiment and returns
print("="*60)
print("CORRELATION ANALYSIS RESULTS")
print("="*60)

correlation_results = {}

# Overall correlation (same-day)
print("\n1. SAME-DAY CORRELATION (Sentiment on day T vs Returns on day T)")
print("-" * 60)
overall_corr = linker.calculate_sentiment_return_correlation(
    aligned_same_day['avg_sentiment'],
    aligned_same_day['daily_return']
)

correlation_results['same_day'] = {
    'data': aligned_same_day,
    'correlation': overall_corr
}

print(f"Pearson Correlation: {overall_corr['correlation']:.4f}")
if not np.isnan(overall_corr['p_value']):
    print(f"P-value: {overall_corr['p_value']:.4e}")
    significance = "***" if overall_corr['p_value'] < 0.001 else "**" if overall_corr['p_value'] < 0.01 else "*" if overall_corr['p_value'] < 0.05 else ""
    print(f"Significance: {significance}")
print(f"Positive returns correlation: {overall_corr['positive_correlation']:.4f}")
print(f"Negative returns correlation: {overall_corr['negative_correlation']:.4f}")
print(f"Number of observations: {len(aligned_same_day):,}")

# Overall correlation (lagged)
print("\n2. LAGGED CORRELATION (Sentiment on day T vs Returns on day T+1)")
print("-" * 60)
lagged_corr = linker.calculate_sentiment_return_correlation(
    aligned_lagged['avg_sentiment'],
    aligned_lagged['daily_return']
)

correlation_results['lagged'] = {
    'data': aligned_lagged,
    'correlation': lagged_corr
}

print(f"Pearson Correlation: {lagged_corr['correlation']:.4f}")
if not np.isnan(lagged_corr['p_value']):
    print(f"P-value: {lagged_corr['p_value']:.4e}")
    significance = "***" if lagged_corr['p_value'] < 0.001 else "**" if lagged_corr['p_value'] < 0.01 else "*" if lagged_corr['p_value'] < 0.05 else ""
    print(f"Significance: {significance}")
print(f"Positive returns correlation: {lagged_corr['positive_correlation']:.4f}")
print(f"Negative returns correlation: {lagged_corr['negative_correlation']:.4f}")
print(f"Number of observations: {len(aligned_lagged):,}")


In [ ]:
# Per-stock correlation analysis
print("\n3. PER-STOCK CORRELATION ANALYSIS (Same-Day)")
print("-" * 60)

stock_correlations = []
for ticker in available_stocks:
    stock_data = aligned_same_day[aligned_same_day['stock'] == ticker]
    if len(stock_data) > 10:  # Need sufficient data points
        corr_stats = linker.calculate_sentiment_return_correlation(
            stock_data['avg_sentiment'],
            stock_data['daily_return']
        )
        
        stock_correlations.append({
            'stock': ticker,
            'correlation': corr_stats['correlation'],
            'p_value': corr_stats['p_value'],
            'n_observations': len(stock_data),
            'mean_sentiment': stock_data['avg_sentiment'].mean(),
            'mean_return': stock_data['daily_return'].mean()
        })

stock_corr_df = pd.DataFrame(stock_correlations)
stock_corr_df = stock_corr_df.sort_values('correlation', ascending=False)

print("\nStock-level correlations:")
print(stock_corr_df.to_string(index=False))

correlation_results['by_stock'] = stock_corr_df


## 10. Visualizations


In [ ]:
# Create visualizations
figures_dir = project_root / 'notebooks' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

# 1. Scatter plot: Sentiment vs Returns (Same-Day)
print("Creating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Correlation Analysis: News Sentiment vs Stock Returns', fontsize=16, fontweight='bold')

# Scatter plot - Same day
ax1 = axes[0, 0]
scatter1 = ax1.scatter(
    aligned_same_day['avg_sentiment'],
    aligned_same_day['daily_return'] * 100,  # Convert to percentage
    alpha=0.5,
    s=20,
    c=aligned_same_day['article_count'],
    cmap='viridis'
)
ax1.set_xlabel('Average Daily Sentiment Score', fontsize=12)
ax1.set_ylabel('Daily Return (%)', fontsize=12)
ax1.set_title(f'Same-Day Correlation\nr = {overall_corr["correlation"]:.4f}, n = {len(aligned_same_day):,}', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.colorbar(scatter1, ax=ax1, label='Articles per Day')

# Scatter plot - Lagged
ax2 = axes[0, 1]
scatter2 = ax2.scatter(
    aligned_lagged['avg_sentiment'],
    aligned_lagged['daily_return'] * 100,
    alpha=0.5,
    s=20,
    c=aligned_lagged['article_count'],
    cmap='viridis'
)
ax2.set_xlabel('Average Daily Sentiment Score', fontsize=12)
ax2.set_ylabel('Daily Return (%) (Next Day)', fontsize=12)
ax2.set_title(f'Lagged Correlation (1-day)\nr = {lagged_corr["correlation"]:.4f}, n = {len(aligned_lagged):,}', fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax2.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.colorbar(scatter2, ax=ax2, label='Articles per Day')

# Correlation by stock
ax3 = axes[1, 0]
colors = ['green' if x > 0 else 'red' for x in stock_corr_df['correlation']]
bars = ax3.barh(stock_corr_df['stock'], stock_corr_df['correlation'], color=colors, alpha=0.7)
ax3.set_xlabel('Correlation Coefficient', fontsize=12)
ax3.set_ylabel('Stock', fontsize=12)
ax3.set_title('Correlation by Stock (Same-Day)', fontsize=11)
ax3.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax3.grid(True, alpha=0.3, axis='x')
for i, (stock, corr) in enumerate(zip(stock_corr_df['stock'], stock_corr_df['correlation'])):
    ax3.text(corr + (0.01 if corr > 0 else -0.01), i, f'{corr:.3f}', 
             va='center', ha='left' if corr > 0 else 'right', fontsize=9)

# Distribution of sentiment scores
ax4 = axes[1, 1]
ax4.hist(aligned_same_day['avg_sentiment'], bins=50, alpha=0.7, edgecolor='black')
ax4.set_xlabel('Average Daily Sentiment Score', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Distribution of Daily Sentiment Scores', fontsize=11)
ax4.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='Neutral')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / 'sentiment_return_correlation.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: {figures_dir / 'sentiment_return_correlation.png'}")
plt.show()


In [ ]:
# Interactive Plotly visualization
print("Creating interactive visualization...")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Same-Day Correlation: Sentiment vs Returns',
        'Lagged Correlation: Sentiment vs Next-Day Returns',
        'Correlation by Stock',
        'Time Series: Sentiment and Returns'
    ),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": True}]]
)

# Same-day scatter
fig.add_trace(
    go.Scatter(
        x=aligned_same_day['avg_sentiment'],
        y=aligned_same_day['daily_return'] * 100,
        mode='markers',
        name='Same-Day',
        marker=dict(
            size=5,
            color=aligned_same_day['article_count'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Articles", x=0.47, y=0.5, len=0.3)
        ),
        text=[f"Stock: {s}<br>Date: {d}<br>Articles: {a}" 
              for s, d, a in zip(aligned_same_day['stock'], 
                                aligned_same_day['date_only'], 
                                aligned_same_day['article_count'])],
        hovertemplate='Sentiment: %{x:.3f}<br>Return: %{y:.2f}%<br>%{text}<extra></extra>'
    ),
    row=1, col=1
)

# Lagged scatter
fig.add_trace(
    go.Scatter(
        x=aligned_lagged['avg_sentiment'],
        y=aligned_lagged['daily_return'] * 100,
        mode='markers',
        name='Lagged',
        marker=dict(
            size=5,
            color=aligned_lagged['article_count'],
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(title="Articles", x=1.02, y=0.5, len=0.3)
        ),
        text=[f"Stock: {s}<br>Date: {d}<br>Articles: {a}" 
              for s, d, a in zip(aligned_lagged['stock'], 
                                aligned_lagged['date_only'], 
                                aligned_lagged['article_count'])],
        hovertemplate='Sentiment: %{x:.3f}<br>Return: %{y:.2f}%<br>%{text}<extra></extra>'
    ),
    row=1, col=2
)

# Correlation by stock
fig.add_trace(
    go.Bar(
        x=stock_corr_df['stock'],
        y=stock_corr_df['correlation'],
        name='Correlation',
        marker=dict(
            color=stock_corr_df['correlation'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="Correlation", x=1.02, y=0.15, len=0.3)
        ),
        text=[f"{c:.3f}" for c in stock_corr_df['correlation']],
        textposition='outside'
    ),
    row=2, col=1
)

# Time series for one stock (AAPL as example)
aapl_data = aligned_same_day[aligned_same_day['stock'] == 'AAPL'].sort_values('date_only')
fig.add_trace(
    go.Scatter(
        x=aapl_data['date_only'],
        y=aapl_data['avg_sentiment'],
        mode='lines',
        name='Sentiment',
        line=dict(color='blue', width=1)
    ),
    row=2, col=2, secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=aapl_data['date_only'],
        y=aapl_data['daily_return'] * 100,
        mode='lines',
        name='Returns',
        line=dict(color='orange', width=1)
    ),
    row=2, col=2, secondary_y=True
)

# Update axes
fig.update_xaxes(title_text="Average Sentiment Score", row=1, col=1)
fig.update_yaxes(title_text="Daily Return (%)", row=1, col=1)
fig.update_xaxes(title_text="Average Sentiment Score", row=1, col=2)
fig.update_yaxes(title_text="Daily Return (%) (Next Day)", row=1, col=2)
fig.update_xaxes(title_text="Stock", row=2, col=1)
fig.update_yaxes(title_text="Correlation Coefficient", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=2)
fig.update_yaxes(title_text="Sentiment Score", row=2, col=2, secondary_y=False)
fig.update_yaxes(title_text="Return (%)", row=2, col=2, secondary_y=True)

fig.update_layout(
    height=900,
    title_text="Comprehensive Sentiment-Return Correlation Analysis",
    showlegend=True
)

fig.write_html(figures_dir / 'sentiment_return_correlation_interactive.html')
print(f"✓ Saved: {figures_dir / 'sentiment_return_correlation_interactive.html'}")
fig.show()


## 11. Summary and Key Findings


In [ ]:
# Summary of findings
print("="*60)
print("CORRELATION ANALYSIS SUMMARY")
print("="*60)

print("\n📊 KEY METRICS:")
print(f"  • Total news articles analyzed: {len(news_df):,}")
print(f"  • Unique date-stock combinations: {len(daily_sentiment):,}")
print(f"  • Matched observations (same-day): {len(aligned_same_day):,}")
print(f"  • Matched observations (lagged): {len(aligned_lagged):,}")

print("\n📈 CORRELATION STRENGTH:")
print(f"  • Same-Day Correlation: {overall_corr['correlation']:.4f}")
if not np.isnan(overall_corr['p_value']):
    sig_level = "Highly significant" if overall_corr['p_value'] < 0.001 else \
                "Significant" if overall_corr['p_value'] < 0.05 else "Not significant"
    print(f"    P-value: {overall_corr['p_value']:.4e} ({sig_level})")

print(f"  • Lagged Correlation (1-day): {lagged_corr['correlation']:.4f}")
if not np.isnan(lagged_corr['p_value']):
    sig_level = "Highly significant" if lagged_corr['p_value'] < 0.001 else \
                "Significant" if lagged_corr['p_value'] < 0.05 else "Not significant"
    print(f"    P-value: {lagged_corr['p_value']:.4e} ({sig_level})")

print("\n📊 SENTIMENT DISTRIBUTION:")
print(f"  • Average sentiment: {aligned_same_day['avg_sentiment'].mean():.4f}")
print(f"  • Median sentiment: {aligned_same_day['avg_sentiment'].median():.4f}")
print(f"  • Std sentiment: {aligned_same_day['avg_sentiment'].std():.4f}")

print("\n📈 RETURN STATISTICS:")
print(f"  • Average daily return: {aligned_same_day['daily_return'].mean():.4f} ({aligned_same_day['daily_return'].mean()*100:.2f}%)")
print(f"  • Std daily return: {aligned_same_day['daily_return'].std():.4f} ({aligned_same_day['daily_return'].std()*100:.2f}%)")

print("\n🏆 TOP CORRELATIONS BY STOCK:")
top_stocks = stock_corr_df.nlargest(3, 'correlation')
for idx, row in top_stocks.iterrows():
    print(f"  • {row['stock']}: {row['correlation']:.4f} (n={row['n_observations']:,})")

print("\n" + "="*60)
print("Analysis Complete!")
print("="*60)
